# Tin tức (Unstructured Data) — One-layer Ingestion

Notebook điều khiển luồng ingestion 1 tầng cho 2 nguồn `rss`, `html` (đã tắt `vnstock`).

Mỗi nguồn ghi output riêng theo layout:
- `news/<source>/date=<run_date>/PART-000.parquet`
- `news/<source>/date=<run_date>/PART-000.csv`

Partition theo ngày (date=YYYY-MM-DD). Khi rerun cùng ngày có thể truncate partition rồi ghi file mới.

Schema của output thống nhất theo `NEWS_COLUMNS`, phục vụ downstream sentiment analysis.

In [8]:
import os, sys, warnings, threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook
def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass
    else:
        _orig_hook(args)
threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

from pathlib import Path
root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ingestion.common import (
    configure_logging,
    load_dotenv_from_project_root,
 )
from ingestion.unstructured_data import (
    NEWS_COLUMNS,
    NewsIngestionConfig,
    ingest_news,
    validate_news_df,
 )

configure_logging()
load_dotenv_from_project_root()
print("[OK] setup (vnstock disabled mode)")

2026-04-18 01:41:54 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env
2026-04-18 01:41:54 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env
2026-04-18 01:41:55 [INFO] API key setup completed
2026-04-18 01:41:55 [INFO] Đã gọi register_user từ VNSTOCK_API_KEY.


✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***6437
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
[OK] setup


## Cấu hình

In [ ]:
rate_limit = int(os.getenv("NEWS_RATE_LIMIT_RPM", "60"))

news_cfg = NewsIngestionConfig(
    use_listing_tickers=True,
    listing_exchange_filter=["HSX", "HNX", "UPCOM"],
    max_tickers_per_run=500,
    max_articles_per_source=200,
    rss_max_per_feed=200,
    html_max_per_source=200,
    days_back=7,
    days_back_vnstock=0,
    days_back_rss=7,
    days_back_html=7,
    strict_published_at_days_back=False,
    prefer_rss_html=True,
    rate_limit_rpm=rate_limit,
    enable_vnstock=False,
    enable_rss=True,
    enable_html=True,
    enable_ticker_match=True,
    append_only=False,
    truncate_partition=True,
)

print(f"run_date     : {news_cfg.run_date}")
print(f"news_root    : {news_cfg.news_root}")
print(f"sources.yaml : {news_cfg.resolved_sources_yaml()}")
print(f"listing      : {news_cfg.resolved_listing_parquet()}")
print(f"listing_ok   : {news_cfg.resolved_listing_parquet().is_file()}")
print(f"tickers      : {len(news_cfg.resolved_tickers())}")
print(f"days_back    : vnstock={news_cfg.days_back_vnstock}d rss={news_cfg.days_back_rss}d html={news_cfg.days_back_html}d (fallback={news_cfg.days_back}, strict={news_cfg.strict_published_at_days_back})")
print(f"max_per      : rss={news_cfg.rss_max_per_feed} html={news_cfg.html_max_per_source}")
print(f"focus_mode   : vnstock_enabled={news_cfg.enable_vnstock} rss_enabled={news_cfg.enable_rss} html_enabled={news_cfg.enable_html}")
print(f"rate_limit   : {news_cfg.rate_limit_rpm} rpm")
print(f"write_mode   : append_only={news_cfg.append_only} (truncate_partition={news_cfg.truncate_partition})")
print(f"schema       : {NEWS_COLUMNS}")

run_date     : 2026-04-18
news_root    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news
sources.yaml : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\ingestion\unstructured_data\sources.yaml
listing      : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet
listing_ok   : True
tickers      : 20
days_back    : vnstock=7d rss=7d html=7d (fallback=7, strict=False)
max_per      : rss=200 html=200
prefer_mode  : prefer_rss_html=True
rate_limit   : 60 rpm (api_key=yes)
write_mode   : append_only=False (truncate_partition=True)
schema       : ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


## Chạy

In [10]:
news_paths = ingest_news(news_cfg)
news_paths

2026-04-18 01:41:55 [INFO] ingest_news: prefer_rss_html=True (vnstock treated as best-effort source)
2026-04-18 01:41:55 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env
2026-04-18 01:41:56 [INFO] API key setup completed
2026-04-18 01:41:56 [INFO] Đã gọi register_user từ VNSTOCK_API_KEY.
2026-04-18 01:41:56 [INFO] vnstock news: 20 tickers, max_per=200


✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***6437
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)


2026-04-18 01:41:56 [INFO] vnstock news YEG@kbs: 1 rows
2026-04-18 01:41:57 [INFO] vnstock news YBM@kbs: 1 rows
2026-04-18 01:41:58 [INFO] vnstock news X20@kbs: 1 rows
2026-04-18 01:41:59 [INFO] vnstock news WSS@kbs: 1 rows
2026-04-18 01:42:00 [INFO] vnstock news WCS@kbs: 1 rows
2026-04-18 01:42:02 [INFO] vnstock news VVS@kbs: 1 rows
2026-04-18 01:42:02 [INFO] vnstock news VTZ@kbs: 1 rows
2026-04-18 01:42:03 [INFO] vnstock news VTV@kbs: 1 rows
2026-04-18 01:42:04 [INFO] vnstock news VTP@kbs: 1 rows
2026-04-18 01:42:05 [INFO] vnstock news VTO@kbs: 1 rows
2026-04-18 01:42:06 [INFO] vnstock news VTJ@kbs: 1 rows
2026-04-18 01:42:07 [INFO] vnstock news VTH@kbs: 1 rows
2026-04-18 01:42:08 [INFO] vnstock news VTC@kbs: 1 rows
2026-04-18 01:42:09 [INFO] vnstock news VTB@kbs: 1 rows
2026-04-18 01:42:10 [INFO] vnstock news VSM@kbs: 1 rows
2026-04-18 01:42:11 [INFO] vnstock news VSI@kbs: 1 rows
2026-04-18 01:42:12 [INFO] vnstock news VSH@kbs: 1 rows
2026-04-18 01:42:13 [INFO] vnstock news VSC@kbs:

{'row_counts': {'vnstock': 0, 'rss': 631, 'html': 100},
 'vnstock': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\vnstock\\date=2026-04-18\\PART-000.parquet',
  'csv': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\vnstock\\date=2026-04-18\\PART-000.csv'},
 'rss': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\rss\\date=2026-04-18\\PART-000.parquet',
  'csv': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\rss\\date=2026-04-18\\PART-000.csv'},
 'html': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\html\\date=2026-04-18\\PART-000.parquet',
  'csv': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\html\\date=2026-04-18\\PART-000.csv'}}

## Kiểm tra kết quả

In [ ]:
import pandas as pd

print("=" * 70)
print("OUTPUT FILES")
print("=" * 70)
for key in ["rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    csv_path = info.get("csv", "")
    if not parquet_path:
        print(f"  {key:8s}: (trống)")
        continue
    df = pd.read_parquet(parquet_path)
    print(f"  {key:8s}: {len(df):>5d} dòng")
    print(f"    parquet: {parquet_path}")
    print(f"    csv    : {csv_path}")

print(f"\nRow counts: {news_paths.get('row_counts', {})}")

OUTPUT FILES
  vnstock :     0 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\vnstock\date=2026-04-18\PART-000.parquet
    csv    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\vnstock\date=2026-04-18\PART-000.csv
  rss     :   631 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\rss\date=2026-04-18\PART-000.parquet
    csv    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\rss\date=2026-04-18\PART-000.csv
  html    :   100 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\html\date=2026-04-18\PART-000.parquet
    csv    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\html\date=2026-04-18\PART-000.csv

Row counts: {'vnstock': 0, 'rss': 631, 'html': 100}


## Validation & Data Quality

In [ ]:
# Validate per-source output with NEWS_COLUMNS
for key in ["rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        print(f"{key}: no output")
        continue

    df = pd.read_parquet(parquet_path)
    issues = validate_news_df(df)
    print(f"{key}: {len(df)} rows")
    if issues:
        print("  ⚠ issues:")
        for i in issues:
            print(f"    - {i}")
    else:
        print("  ✓ validation passed")
    print(f"  columns: {list(df.columns)}")

vnstock: 0 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']
rss: 631 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']
html: 100 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


In [ ]:
# Quick preview for sentiment-ready fields
for key in ["rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        continue
    df = pd.read_parquet(parquet_path)
    print(f"\n[{key}] preview")
    print(df[["article_id", "source", "ticker", "title", "body_text", "published_at"]].head(3))


[vnstock] preview
Empty DataFrame
Columns: [article_id, source, ticker, title, body_text, published_at]
Index: []

[rss] preview
                                          article_id  \
0  85fa78bf47114ddd777935130b0fe24bb792c2c68bc00d...   
1  776b0edfe50e764c0ba5ef222eac1ac6273da5b3856ea9...   
2  253a8662673287a5ae9a6de9a8e55272602856739c58e5...   

                     source ticker  \
0  vnexpress_kinh_doanh_rss    NaN   
1  vnexpress_kinh_doanh_rss    NaN   
2  vnexpress_kinh_doanh_rss    NaN   

                                               title body_text  \
0  Thách thức nhiên liệu vừa đắt vừa thiếu của hà...             
1  Nhiều ngân hàng dừng chuyển tiền nhanh trên 50...             
2  Giá dầu giảm mạnh, vàng bật tăng sau tin eo bi...             

               published_at  
0 2026-04-17 17:36:04+00:00  
1 2026-04-17 17:26:22+00:00  
2 2026-04-17 13:38:42+00:00  

[html] preview
                                          article_id          source ticker  \
0  85fa78bf4

In [ ]:
# Quality report: row count, nulls, published_at range for RSS + HTML
import pandas as pd

sources = ["rss", "html"]
main_cols = ["article_id", "title", "url", "published_at"]

report_rows = []
for src in sources:
    info = news_paths.get(src, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        report_rows.append(
            {
                "source": src,
                "rows": 0,
                "min_published_at": None,
                "max_published_at": None,
                **{f"null_{c}": None for c in main_cols},
            }
        )
        continue

    df = pd.read_parquet(parquet_path)
    row = {"source": src, "rows": int(len(df))}
    for col in main_cols:
        row[f"null_{col}"] = int(df[col].isna().sum()) if col in df.columns else int(len(df))

    if "published_at" in df.columns:
        pub = pd.to_datetime(df["published_at"], errors="coerce", utc=True)
        if pub.notna().any():
            row["min_published_at"] = str(pub.min())
            row["max_published_at"] = str(pub.max())
        else:
            row["min_published_at"] = None
            row["max_published_at"] = None
    else:
        row["min_published_at"] = None
        row["max_published_at"] = None

    report_rows.append(row)

quality_df = pd.DataFrame(report_rows)
print("QUALITY REPORT")
display(quality_df)

counts = {r["source"]: int(r["rows"]) for r in report_rows}
if counts.get("rss", 0) == 0 or counts.get("html", 0) == 0:
    print("⚠ Warning: RSS hoặc HTML đang rỗng — cần kiểm tra lại feed/selector.")
else:
    print("✓ RSS + HTML balance looks acceptable for this run.")

QUALITY REPORT


,source,rows,null_article_id,null_title,null_url,null_published_at,min_published_at,max_published_at
0,vnstock,0,0,0,0,0,NaN,NaN
1,rss,631,0,0,0,0,2026-04-11 02:30:00+00:00,2026-04-17 17:36:04+00:00
2,html,100,0,0,0,100,NaN,NaN


⚠ Warning: vnstock đang thấp nhưng RSS/HTML đang cao — đây là trạng thái chấp nhận được khi vnstock chạy best-effort.
